# Exploração da demanda diária por bicicletas compartilhadas

## Contexto e objetivo

Esta etapa descreve a demanda diária do serviço de bicicletas compartilhadas e explora sua relação com o clima e o calendário. A pergunta central é: **como as condições meteorológicas e as características do calendário se relacionam com o número diário de aluguéis e podem ajudar a planejar a disponibilidade de bicicletas?**

Os resultados descritivos orientam perguntas para a futura análise estatística e o planejamento geral do serviço. A base diária não permite estimar a demanda por estação ou definir a redistribuição entre estações. Diferenças observadas não demonstram significância estatística, e associação não comprova causalidade.

## Dataset e unidade de observação

Utiliza-se o **Bike Sharing**, da UCI, referente ao sistema Capital Bikeshare, em Washington, nos anos de **2011 e 2012**. O arquivo diário `day.csv` é lido de `../dados/brutos/day.csv`. Cada linha representa **um dia**, e `cnt` é o **total de aluguéis realizados nesse dia**, incluindo `casual` e `registered`.

Fonte oficial: [UCI — Bike Sharing](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset). Referência: Fanaee-T, H. (2013). *Bike Sharing* [Dataset]. DOI: https://doi.org/10.24432/C5W894.

A inspeção inicial abaixo apresenta uma amostra, as dimensões, os tipos e o resumo numérico da base.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../dados/brutos/day.csv')
df.head()

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


In [3]:
df.shape

(731, 16)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     731 non-null    int64  
 1   dteday      731 non-null    str    
 2   season      731 non-null    int64  
 3   yr          731 non-null    int64  
 4   mnth        731 non-null    int64  
 5   holiday     731 non-null    int64  
 6   weekday     731 non-null    int64  
 7   workingday  731 non-null    int64  
 8   weathersit  731 non-null    int64  
 9   temp        731 non-null    float64
 10  atemp       731 non-null    float64
 11  hum         731 non-null    float64
 12  windspeed   731 non-null    float64
 13  casual      731 non-null    int64  
 14  registered  731 non-null    int64  
 15  cnt         731 non-null    int64  
dtypes: float64(4), int64(11), str(1)
memory usage: 98.6 KB


### Leitura do resumo numérico inicial

Em `describe()`, `count` indica a quantidade de observações não ausentes; `mean` é a média; `50%` é a mediana; e `std` é o desvio padrão amostral, que descreve a dispersão entre dias. `min`, `max`, `25%` e `75%` indicam extremos e quartis. Para `cnt`, essas medidas descrevem aluguéis diários, enquanto `count = 731` significa **731 dias observados**, não 731 aluguéis.

A tabela também inclui identificadores e códigos categóricos porque estão armazenados como números. As médias de `instant`, `season`, `weekday` ou `weathersit`, por exemplo, não devem ser interpretadas automaticamente como medidas quantitativas do fenômeno.

In [5]:
df.describe()

,instant,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
count,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000
mean,366.000000,2.496580,0.500684,6.519836,0.028728,2.997264,0.683995,1.395349,0.495385,0.474354,0.627894,0.190486,848.176471,3656.172367,4504.348837
std,211.165812,1.110807,0.500342,3.451913,0.167155,2.004787,0.465233,0.544894,0.183051,0.162961,0.142429,0.077498,686.622488,1560.256377,1937.211452
min,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.059130,0.079070,0.000000,0.022392,2.000000,20.000000,22.000000
25%,183.500000,2.000000,0.000000,4.000000,0.000000,1.000000,0.000000,1.000000,0.337083,0.337842,0.520000,0.134950,315.500000,2497.000000,3152.000000
50%,366.000000,3.000000,1.000000,7.000000,0.000000,3.000000,1.000000,1.000000,0.498333,0.486733,0.626667,0.180975,713.000000,3662.000000,4548.000000
75%,548.500000,3.000000,1.000000,10.000000,0.000000,5.000000,1.000000,2.000000,0.655417,0.608602,0.730209,0.233214,1096.000000,4776.500000,5956.000000
max,731.000000,4.000000,1.000000,12.000000,1.000000,6.000000,1.000000,3.000000,0.861667,0.840896,0.972500,0.507463,3410.000000,6946.000000,8714.000000


## Preparação das datas

`dteday` é convertida para um tipo de data, com `errors="raise"` para interromper a conversão se houver um valor inválido. A base é ordenada cronologicamente e o índice é reiniciado. A saída posterior confirma 731 datas não ausentes do tipo `datetime64[us]`.

As colunas originais são mantidas: apenas o tipo de `dteday` e a ordenação são ajustados. Os rótulos e as variáveis de calendário serão adicionados em novas colunas, sem substituir os códigos originais.

In [6]:
df["dteday"] = pd.to_datetime(df["dteday"], errors = "raise")
df = df.sort_values("dteday").reset_index(drop = True)

In [7]:
df.head()

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


In [8]:
df["dteday"].info()

<class 'pandas.Series'>
RangeIndex: 731 entries, 0 to 730
Series name: dteday
Non-Null Count  Dtype         
--------------  -----         
731 non-null    datetime64[us]
dtypes: datetime64[us](1)
memory usage: 5.8 KB


## Auditoria da base

A auditoria abaixo verifica a integridade dos registros antes dos agrupamentos. A continuidade é avaliada entre a menor e a maior data da base, comparando as datas registradas com uma sequência diária esperada. Essa verificação não avalia datas fora desse intervalo.

In [9]:
datas_esperadas = pd.date_range(df["dteday"].min(), df["dteday"].max(), freq = "D")

auditoria = pd.Series({
    "registros": len(df),
    "colunas": df.shape[1],
    "valores_ausentes": int(df.isna().sum().sum()),
    "linhas_duplicadas": int(df.duplicated().sum()),
    "datas_duplicadas": int(df["dteday"].duplicated().sum()),
    "dias_sem_registro": len(datas_esperadas.difference(df["dteday"])),
    "totais_inconsistentes": int((df["casual"] + df["registered"] != df["cnt"]).sum()),
    "contagens_negativas": int((df[["casual", "registered", "cnt"]] < 0).sum().sum())
})

auditoria

registros                731
colunas                   16
valores_ausentes           0
linhas_duplicadas          0
datas_duplicadas           0
dias_sem_registro          0
totais_inconsistentes      0
contagens_negativas        0
dtype: int64

### Resultado das verificações executadas

A saída registra **731 linhas e 16 colunas**:

| Verificação | Significado | Resultado |
|---|---|---|
| Valores ausentes | Células sem valor em qualquer coluna | 0 |
| Linhas duplicadas | Repetições considerando todas as colunas | 0 |
| Datas duplicadas | Mais de um registro para a mesma data | 0 |
| Dias sem registro | Datas ausentes entre os extremos observados | 0 |
| Totais inconsistentes | Dias em que `casual + registered != cnt` | 0 |
| Contagens negativas | Valores negativos nas três colunas de aluguéis | 0 |

Não foram identificados problemas nessas verificações. Isso não garante ausência de outros erros de medição. A inspeção seguinte lista os códigos efetivamente presentes nas variáveis categóricas; não é uma validação exaustiva de todos os domínios.

In [10]:
categoricas = [
    "season", "yr", "mnth", "holiday",
    "weekday", "workingday", "weathersit"
]

for coluna in categoricas:
    print(coluna, sorted(df[coluna].unique()))

season [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
yr [np.int64(0), np.int64(1)]
mnth [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
holiday [np.int64(0), np.int64(1)]
weekday [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
workingday [np.int64(0), np.int64(1)]
weathersit [np.int64(1), np.int64(2), np.int64(3)]


### Rótulos categóricos

A inspeção encontrou `season` de 1 a 4, `yr` em 0 e 1, `mnth` de 1 a 12, `holiday` e `workingday` em 0 e 1, `weekday` de 0 a 6 e `weathersit` em 1, 2 e 3. São códigos de categorias, não quantidades contínuas.

O mapeamento cria `estacao`, `ano`, `mes`, `feriado`, `dia_semana`, `tipo_dia` e `condicao_climatica`, preservando suas colunas de origem. Para `season`, usa-se **1 = inverno, 2 = primavera, 3 = verão e 4 = outono**, conforme a [documentação atual da UCI](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset); a amostra de janeiro com código 1 é compatível com essa correspondência no hemisfério norte.

`yr` distingue 2011 (0) de 2012 (1); `weekday` começa no domingo (0) e termina no sábado (6). `workingday = 1` identifica dias que não são fim de semana nem feriado; `workingday = 0` reúne fins de semana e feriados. O rótulo meteorológico 4 está previsto no mapeamento, mas esse código não aparece na saída da base diária.

In [11]:
rotulos = {
    "season": {
        "coluna": "estacao",
        "valores": {
            1: "Inverno",
            2: "Primavera",
            3: "Verão",
            4: "Outono"
        }
    },
    "yr": {
        "coluna": "ano",
        "valores": {
            0: "2011",
            1: "2012"
        }
    },
    "mnth": {
        "coluna": "mes",
        "valores": {
            1: "Janeiro",
            2: "Fevereiro",
            3: "Março",
            4: "Abril",
            5: "Maio",
            6: "Junho",
            7: "Julho",
            8: "Agosto",
            9: "Setembro",
            10: "Outubro",
            11: "Novembro",
            12: "Dezembro"
        }
    },
    "holiday": {
        "coluna": "feriado",
        "valores": {
            0: "Não",
            1: "Sim"
        }
    },
    "weekday": {
        "coluna": "dia_semana",
        "valores": {
            0: "Domingo",
            1: "Segunda-feira",
            2: "Terça-feira",
            3: "Quarta-feira",
            4: "Quinta-feira",
            5: "Sexta-feira",
            6: "Sábado"
        }
    },
    "workingday": {
        "coluna": "tipo_dia",
        "valores": {
            0: "Fim de semana ou feriado",
            1: "Dia útil"
        }
    },
    "weathersit": {
        "coluna": "condicao_climatica",
        "valores": {
            1: "Tempo aberto ou parcialmente nublado",
            2: "Névoa ou neblina, com possíveis nuvens",
            3: "Chuva leve ou neve leve",
            4: "Chuva forte ou neve e condições severas"
        }
    }
}

for original, configuracao in rotulos.items():
    df[configuracao["coluna"]] = df[original].map(
        configuracao["valores"]
    )

## Estatísticas descritivas da demanda

O resumo específico de `cnt` apresenta **731 dias**, média de **4.504,35 aluguéis por dia**, mediana de **4.548** e desvio padrão de **1.937,21 aluguéis**. A mediana divide os dias ordenados pela demanda em duas metades; o desvio padrão expressa a variação diária, não a incerteza da média.

O mínimo observado é **22** e o máximo é **8.714** aluguéis. Esses extremos não foram excluídos nem classificados como erros nesta exploração.

In [12]:
df["cnt"].describe()

count     731.000000
mean     4504.348837
std      1937.211452
min        22.000000
25%      3152.000000
50%      4548.000000
75%      5956.000000
max      8714.000000
Name: cnt, dtype: float64

## Comparação por tipo de dia

A comparação usa médias diárias porque a base contém **500 dias úteis** e **231 dias de fim de semana ou feriado**. Um total acumulado depende tanto da demanda por dia quanto da quantidade de dias em cada grupo; a média divide os aluguéis pelo número de dias e permite comparar a intensidade diária. A tabela abaixo não calcula totais por tipo de dia.

A média é **4.584,82** aluguéis em dias úteis e **4.330,17** nos demais dias; as medianas são **4.582** e **4.459**, respectivamente. Os desvios padrão são **1.878,42** e **2.052,14**. A média maior em dias úteis é uma diferença descritiva: não comprova significância estatística nem deslocamento para o trabalho, pois os motivos das viagens não são registrados.

In [13]:
df.groupby("workingday")["cnt"].agg(
    ["count", "mean", "median", "std", "min", "max"]
) 

,count,mean,median,std,min,max
workingday,,,,,,
0,231,4330.168831,4459.0,2052.141245,605,8714
1,500,4584.820000,4582.0,1878.415583,22,8362


## Comparação por dia da semana

O agrupamento procura identificar diferenças no ciclo semanal. Há **104 ou 105 dias** por categoria, com quantidades de observações semelhantes. A maior média é a de sexta-feira (**4.690,29**), seguida de quinta-feira (**4.667,26**); a menor é a de domingo (**4.228,83**). O sábado apresenta o maior desvio padrão (**2.196,69**), indicando maior dispersão diária nesse grupo.

Esta tabela reúne 2011 e 2012. Portanto, descreve um padrão semanal agregado e não separa mudanças entre anos; as comparações mensais seguintes mantêm os anos separados para examinar sazonalidade e crescimento.

In [14]:
resumo_semana = df.groupby(["weekday", "dia_semana"])["cnt"].agg(
    dias_observados = "count",
    media_diaria = "mean",
    mediana_diaria = "median",
    desvio_padrao = "std"
).reset_index().sort_values("weekday")

resumo_semana.round(2)

,weekday,dia_semana,dias_observados,media_diaria,mediana_diaria,desvio_padrao
0,0,Domingo,105,4228.83,4334.0,1872.50
1,1,Segunda-feira,105,4338.12,4359.0,1793.07
2,2,Terça-feira,104,4510.66,4576.5,1826.91
3,3,Quarta-feira,104,4548.54,4642.5,2038.10
4,4,Quinta-feira,104,4667.26,4721.0,1939.43
5,5,Sexta-feira,104,4690.29,4601.5,1874.62
6,6,Sábado,105,4550.54,4521.0,2196.69


## Comparação por mês, ano e tipo de dia

São criadas `ano_calendario` e `mes_num` a partir da data, mantendo `yr` e `mnth`. O agrupamento combina ano, mês e tipo de dia para observar variações sazonais sem misturar o nível de demanda de 2011 com o de 2012.

`dias_observados` informa o número de dias em cada combinação; média, mediana e desvio padrão descrevem os aluguéis por dia nesse grupo. Por exemplo, janeiro de 2011 tem **11 dias não úteis** e **20 dias úteis**, com médias de **1.016,36** e **1.350,45**, respectivamente. Quantidades de dias e quantidades de aluguéis são medidas distintas.

In [15]:
df["ano_calendario"] = df["dteday"].dt.year
df["mes_num"] = df["dteday"].dt.month

resumo_mensal = df.groupby(["ano_calendario", "mes_num", "workingday", "tipo_dia"])["cnt"].agg(
    dias_observados = "count",
    media_diaria = "mean",
    mediana_diaria = "median",
    desvio_padrao = "std"
).reset_index()

resumo_mensal.round(2)

,ano_calendario,mes_num,workingday,tipo_dia,dias_observados,media_diaria,mediana_diaria,desvio_padrao
0,2011,1,0,Fim de semana ou feriado,11,1016.36,986.0,138.83
1,2011,1,1,Dia útil,20,1350.45,1418.5,408.87
2,2011,2,0,Fim de semana ou feriado,9,1623.78,1623.0,423.46
3,2011,2,1,Dia útil,19,1768.47,1708.0,389.12
4,2011,3,0,Fim de semana ou feriado,8,2126.00,2274.5,739.31
5,2011,3,1,Dia útil,23,2045.09,2046.0,488.14
6,2011,4,0,Fim de semana ou feriado,10,3205.50,3187.5,1237.19
7,2011,4,1,Dia útil,20,3140.75,3235.5,964.88
8,2011,5,0,Fim de semana ou feriado,10,4446.90,4606.5,714.81
9,2011,5,1,Dia útil,21,4350.10,4433.0,509.19


### Gráfico das médias mensais

O gráfico compara a trajetória das médias diárias de dias úteis e não úteis, com um painel para cada ano. Os pontos representam médias de grupos mensais; as linhas ajudam a acompanhar a sequência dos meses, sem representar todos os valores diários. As informações de quantidade de dias e desvio padrão ficam disponíveis ao passar o cursor.

As tabelas que alimentam o gráfico mostram níveis mais baixos no início do ano e mais altos em meses intermediários, além de níveis maiores em 2012. A relação entre tipos de dia varia conforme o mês; as linhas não representam intervalos de confiança.

In [16]:
import plotly.express as px

fig = px.line(
    resumo_mensal,
    x="mes_num",
    y="media_diaria",
    color="tipo_dia",
    facet_col="ano_calendario",
    markers=True,
    labels={
        "mes_num": "Mês",
        "media_diaria": "Média diária de aluguéis",
        "tipo_dia": "Tipo de dia",
        "ano_calendario": "Ano"
    },
    title="Demanda média diária por mês e tipo de dia",
    hover_data=["dias_observados", "desvio_padrao"]
)

fig.update_xaxes(dtick=1)
fig.show()

### Diferenças mensais entre tipos de dia

A tabela reorganiza as médias e calcula `media_util - media_nao_util`. A diferença percentual usa a média dos dias não úteis como referência: valores positivos indicam média maior nos dias úteis; valores negativos indicam média maior nos dias não úteis.

Em setembro de 2011, a diferença é **−660,10 aluguéis por dia (−14,02%)**. Em dezembro de 2012, é **1.230,02 (38,47%)**. Em setembro de 2012, as médias são próximas: diferença de **23,46 (0,32%)**. Esses exemplos mostram que a diferença agregada por tipo de dia não se repete de forma uniforme em todos os meses.

In [17]:
comparacao_mensal = resumo_mensal.pivot(
    index=["ano_calendario", "mes_num"],
    columns="workingday",
    values="media_diaria"
)

comparacao_mensal = comparacao_mensal.rename(columns={
    0: "media_nao_util",
    1: "media_util"
})

comparacao_mensal["diferenca_alugueis"] = (
    comparacao_mensal["media_util"]
    - comparacao_mensal["media_nao_util"]
)

comparacao_mensal["diferenca_percentual"] = (
    comparacao_mensal["diferenca_alugueis"]
    / comparacao_mensal["media_nao_util"]
    * 100
)

comparacao_mensal.round(2)

workingday              media_nao_util  media_util  diferenca_alugueis  \
ano_calendario mes_num                                                   
2011           1               1016.36     1350.45              334.09   
               2               1623.78     1768.47              144.70   
               3               2126.00     2045.09              -80.91   
               4               3205.50     3140.75              -64.75   
               5               4446.90     4350.10              -96.80   
               6               5005.50     4703.09             -302.41   
               7               4811.00     4421.00             -390.00   
               8               3820.25     4614.30              794.05   
               9               4709.33     4049.24             -660.10   
               10              4026.27     3961.10              -65.17   
               11              3354.40     3431.15               76.75   
               12              2376.90     3026.38              649.48   
2012           1               2712.45     3345.35              632.90   
               2               2859.33     3870.15             1010.82   
               3               4983.22     5455.73              472.51   
               4               5720.40     5851.00              130.60   
               5               6820.22     6112.86             -707.36   
               6               7174.78     6583.67             -591.11   
               7               6059.70     6810.00              750.30   
               8               6106.62     7202.17             1095.55   
               9               7270.91     7294.37               23.46   
               10              6436.22     6405.23              -30.99   
               11              4732.60     5266.90              534.30   
               12              3197.18     4427.20             1230.02   

workingday              diferenca_percentual  
ano_calendario mes_num                        
2011           1                       32.87  
               2                        8.91  
               3                       -3.81  
               4                       -2.02  
               5                       -2.18  
               6                       -6.04  
               7                       -8.11  
               8                       20.79  
               9                      -14.02  
               10                      -1.62  
               11                       2.29  
               12                      27.32  
2012           1                       23.33  
               2                       35.35  
               3                        9.48  
               4                        2.28  
               5                      -10.37  
               6                       -8.24  
               7                       12.38  
               8                       17.94  
               9                        0.32  
               10                      -0.48  
               11                      11.29  
               12                      38.47

## Evolução anual e perfis de utilização

O resumo anual diferencia a extensão do período, o total acumulado e a média diária. Em **2011**, há **365 dias**, **1.243.103 aluguéis** e média de **3.405,76 por dia**. Em **2012**, há **366 dias**, **2.049.576 aluguéis** e média de **5.599,93 por dia**. Comparar as médias evita atribuir ao crescimento da demanda a diferença de um dia entre os anos.

Os aluguéis de cadastrados predominam nos dois anos: suas médias diárias são **2.728,36** e **4.581,45**, enquanto as de casuais são **677,40** e **1.018,48**. Esses valores descrevem perfis de utilização por categoria de cadastro. `casual` e `registered` contam **aluguéis**, não pessoas distintas, novos cadastros, assinantes ativos ou motivos de viagens. Não há nesta etapa comparação desses perfis por tipo de dia.

In [18]:
resumo_anual = (
    df.groupby("ano_calendario")
    .agg(
        dias_observados=("cnt", "count"),
        total_alugueis=("cnt", "sum"),
        media_diaria=("cnt", "mean"),
        media_casuais=("casual", "mean"),
        media_cadastrados=("registered", "mean")
    )
)

resumo_anual.round(2)


,dias_observados,total_alugueis,media_diaria,media_casuais,media_cadastrados
ano_calendario,,,,,
2011,365,1243103,3405.76,677.40,2728.36
2012,366,2049576,5599.93,1018.48,4581.45


### Crescimento das médias por perfil

O cálculo seguinte compara as médias de 2012 às de 2011. A média total cresce **64,43%**, a de aluguéis casuais **50,35%** e a de cadastrados **67,92%**. O crescimento maior dos aluguéis de cadastrados não permite separar aumento do número de usuários de aumento de viagens por usuário.

A base não mede trânsito, incentivos, quantidade de assinantes ou expansão da rede. Explicações do crescimento baseadas nesses fatores permanecem hipóteses, sem comprovação pelas saídas disponíveis.

In [19]:
medias = [
    "media_diaria",
    "media_casuais",
    "media_cadastrados"
]

crescimento_percentual = (
    resumo_anual.loc[2012, medias]
    / resumo_anual.loc[2011, medias]
    - 1
) * 100

crescimento_percentual.round(2)

media_diaria         64.43
media_casuais        50.35
media_cadastrados    67.92
dtype: float64

### Comparação do mesmo mês entre anos

Comparar cada mês de 2011 ao correspondente de 2012 ajuda a distinguir diferenças anuais da variação ao longo do calendário. A média de 2012 é maior em todos os meses. O crescimento percentual varia de **41,33% em junho** a **157,44% em março**.

Em 2011, a maior média mensal ocorre em junho (**4.783,73**); em 2012, em setembro (**7.285,77**). A separação dos anos mostra variação sazonal e crescimento, mas não identifica as causas dessas mudanças.

In [20]:
comparacao_anos = (
    df.groupby(["mes_num", "ano_calendario"])["cnt"]
    .mean()
    .unstack("ano_calendario")
)

comparacao_anos["crescimento_percentual"] = (
    comparacao_anos[2012] / comparacao_anos[2011] - 1
) * 100

comparacao_anos.round(2)

ano_calendario,2011,2012,crescimento_percentual
mes_num,,,
1,1231.90,3120.77,153.33
2,1721.96,3556.45,106.53
3,2065.97,5318.55,157.44
4,3162.33,5807.47,83.64
5,4381.32,6318.23,44.21
6,4783.73,6761.00,41.33
7,4559.39,6567.97,44.05
8,4409.39,6919.45,56.93
9,4247.27,7285.77,71.54


## Clima e demanda

### Distribuição das variáveis meteorológicas por ano

`temp`, `atemp`, `hum` e `windspeed` são **variáveis normalizadas**, conforme a [fonte oficial](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset). Os valores abaixo permanecem nessa escala; não são apresentados como graus Celsius, porcentagem de umidade ou velocidade em unidade física. Uma conversão exige verificar a transformação aplicável ao arquivo diário.

O resumo anual inclui `temp`, `hum` e `windspeed`, com média, mediana, desvio padrão e extremos. A média de `temp` passa de **0,4867** para **0,5041**; a de `hum`, de **0,6437** para **0,6122**; e a de `windspeed`, de **0,1914** para **0,1896**. Essas diferenças descrevem o clima observado em cada ano e não demonstram que ele explica o crescimento da demanda. `atemp` aparece no resumo inicial, mas não nesta comparação anual. A relação individual dessas variáveis com `cnt` ainda não foi analisada.

In [21]:
resumo_clima_anual = (
    df.groupby("ano_calendario")[
        ["temp", "hum", "windspeed"]
    ]
    .agg(["mean", "median", "std", "min", "max"])
)

resumo_clima_anual.round(4)

temp                                     hum          \
                  mean  median     std     min     max    mean  median   
ano_calendario                                                           
2011            0.4867  0.4792  0.1896  0.0591  0.8492  0.6437  0.6475   
2012            0.5041  0.5142  0.1761  0.1075  0.8617  0.6122  0.6119   

                                       windspeed                          \
                   std     min     max      mean  median     std     min   
ano_calendario                                                             
2011            0.1487  0.0000  0.9725    0.1914  0.1869  0.0769  0.0224   
2012            0.1342  0.2542  0.9250    0.1896  0.1748  0.0782  0.0466   

                        
                   max  
ano_calendario          
2011            0.5075  
2012            0.4416

### Demanda por condição meteorológica e ano

O agrupamento compara dias de uma mesma categoria meteorológica dentro de cada ano, evitando misturar os diferentes níveis anuais de demanda. Os rótulos usados são resumos das condições descritas pela UCI; `weathersit` representa categorias, não uma escala quantitativa contínua.

| Condição (`weathersit`) | Dias em 2011 | Média diária em 2011 | Dias em 2012 | Média diária em 2012 |
|---|---:|---:|---:|---:|
| Tempo aberto ou parcialmente nublado (1) | 226 | 3.694,99 | 237 | 6.003,73 |
| Névoa ou neblina, com possíveis nuvens (2) | 124 | 3.088,10 | 123 | 4.991,33 |
| Chuva leve ou neve leve (3) | 15 | 1.674,13 | 6 | 2.126,17 |

Em ambos os anos, a média é maior na categoria 1 e menor na categoria 3; dentro de cada categoria, a média de 2012 supera a de 2011. Essas associações podem envolver também calendário e outras diferenças entre períodos.

A categoria 3 tem **poucas observações**, especialmente em 2012: **6 dias**, mediana de **1.598** e desvio padrão de **2.011,65**, frente à média de **2.126,17**. Esse grupo exige cautela pela pequena quantidade de dias e grande dispersão. A categoria 4 não aparece na base diária, portanto não há resultado para ela.

In [22]:
demanda_por_clima = (
    df.groupby(
        ["ano_calendario", "weathersit", "condicao_climatica"]
    )["cnt"]
    .agg(
        dias_observados="count",
        media_diaria="mean",
        mediana_diaria="median",
        desvio_padrao="std"
    )
    .reset_index()
)

demanda_por_clima.round(2)

,ano_calendario,weathersit,condicao_climatica,dias_observados,media_diaria,mediana_diaria,desvio_padrao
0,2011,1,Tempo aberto ou parcialmente nublado,226,3694.99,4103.5,1337.00
1,2011,2,"Névoa ou neblina, com possíveis nuvens",124,3088.10,3380.0,1286.88
2,2011,3,Chuva leve ou neve leve,15,1674.13,1817.0,830.52
3,2012,1,Tempo aberto ou parcialmente nublado,237,6003.73,6398.0,1611.87
4,2012,2,"Névoa ou neblina, com possíveis nuvens",123,4991.33,5191.0,1759.66
5,2012,3,Chuva leve ou neve leve,6,2126.17,1598.0,2011.65


## Síntese e próximos passos

As saídas disponíveis sustentam os seguintes achados descritivos:

- A auditoria não identificou problemas nas verificações realizadas; há 731 dias e consistência entre as contagens dos dois perfis e o total.
- A média diária é maior nos dias úteis, mas a direção da diferença varia por mês. Há também diferenças no ciclo semanal.
- A demanda apresenta variação mensal e médias maiores em 2012 em todos os meses; o crescimento da média anual total é de 64,43%.
- Os aluguéis de cadastrados predominam e sua média cresce proporcionalmente mais que a de casuais, sem permitir inferir quantos usuários adicionais utilizaram o serviço.
- As médias são menores sob chuva leve ou neve leve em ambos os anos, com poucos dias nessa categoria.

Esses padrões podem orientar o planejamento geral da disponibilidade de bicicletas por época do ano e cenário meteorológico. Ainda não foram realizados testes de hipóteses, intervalos de confiança ou regressões; as diferenças não demonstram significância estatística, e as associações não comprovam causalidade.

### Perguntas para a futura análise estatística

1. A diferença entre dias úteis e não úteis permanece ao considerar ano, mês e clima? Qual é sua magnitude e incerteza?
2. Como temperatura, sensação térmica, umidade e vento se associam à demanda diária? `temp` e `atemp` oferecem informação redundante?
3. As diferenças por condição meteorológica persistem após considerar o calendário? Como tratar a pequena quantidade de dias na categoria 3?
4. Qual é a dependência entre dias consecutivos, e como incorporá-la aos futuros testes e intervalos de confiança?
5. Um modelo com clima e calendário prevê datas posteriores melhor que uma referência simples, usando uma separação temporal entre treino e avaliação?

`casual` e `registered` **não devem prever `cnt`**, pois sua soma já é o total e forneceria a resposta ao modelo. Explicações sobre trabalho, trânsito, incentivos, assinantes ou expansão da rede dependem de informações que esta base não mede.

### Lacunas desta etapa

Não há saídas de testes, intervalos ou modelos, nem comparações dos perfis por calendário ou análises da relação de cada variável meteorológica contínua com a demanda. A tabela semanal reúne os dois anos. A última célula de código está vazia e sem execução; não representa uma análise concluída. As células de preparação sem saída explícita são descritas pelas operações presentes no código, sem lhes atribuir resultados adicionais.